# Piper TTS Training — Iu Mienh Voice

Train a custom Piper TTS model for the Iu Mienh language using Bible audio data.

## Prerequisites
1. Upload your `tts-data/` folder as a Kaggle Dataset (wavs/ + metadata.txt)
2. Enable GPU accelerator (Settings → Accelerator → GPU P100 or T4×2)
3. Set persistence to "Files" so checkpoints survive restarts

### Uploading your dataset to Kaggle:
```bash
# On your local machine, zip just what's needed:
cd tts-data
zip -r iu-mienh-tts.zip wavs/ metadata.txt
# Upload via kaggle.com → Datasets → New Dataset → "iu-mienh-tts"
# Or use the Kaggle CLI:
# kaggle datasets create -p . --dir-mode zip
```

After upload, your data will be at: `/kaggle/input/iu-mienh-tts/`

## 1. Install Dependencies

In [ ]:
import os, subprocess, sys
os.environ['DEBIAN_FRONTEND'] = 'noninteractive'

# Kaggle uses Python 3.10 or 3.12 — piper-phonemize needs to be built from source
PY_VER = f'{sys.version_info.major}.{sys.version_info.minor}'
print(f'Python version: {PY_VER}')

# Install core training dependencies (torch is pre-installed on Kaggle GPU)
!pip install -q pytorch-lightning==1.9.5 onnxruntime librosa tensorboard

# Install piper-phonemize from GitHub release (pre-built wheel)
# Try the release wheel first, fall back to building from source
PHONEMIZE_URL = 'https://github.com/rhasspy/piper-phonemize/releases/download/2024.11.14-2/piper_phonemize-1.2.0-cp310-cp310-manylinux_2_28_x86_64.whl'
if PY_VER == '3.12':
    # For Python 3.12, install espeak-ng and use a simpler approach
    !apt-get install -y -qq espeak-ng libespeak-ng-dev
    # Build piper-phonemize from source for Python 3.12
    !pip install -q piper-phonemize --no-binary :all: 2>/dev/null || true
elif PY_VER == '3.10':
    !pip install -q {PHONEMIZE_URL}
else:
    !pip install -q piper-phonemize || true

# Clone piper and install training code
!pip install -q cython
!rm -rf /tmp/piper && git clone --depth 1 https://github.com/rhasspy/piper.git /tmp/piper

# Install piper_train without dependency checks (we handle deps above)
!cd /tmp/piper/src/python && pip install --no-deps -e .

# Build monotonic alignment (Cython extension required for VITS)
!cd /tmp/piper/src/python && bash build_monotonic_align.sh

# Verify import works
try:
    from piper_train.vits.lightning import VitsModel
    print('\n✅ Dependencies installed — piper_train imported successfully')
except ImportError as e:
    print(f'\n⚠️  Import warning: {e}')
    print('Training may still work if piper-phonemize installed correctly')

## 2. Prepare Dataset

Convert our `metadata.txt` (Piper format: `filename|text`) into LJSpeech format that `piper_train.preprocess` expects.

In [ ]:
from pathlib import Path
import shutil, glob

# === CONFIGURATION ===
WORK_DIR = Path('/kaggle/working/iu-mienh')
DATASET_DIR = WORK_DIR / 'dataset'
OUTPUT_DIR = WORK_DIR / 'output'

DATASET_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Find the dataset files (tar.gz may extract into subdirectory)
INPUT_BASE = Path('/kaggle/input')
metadata_candidates = list(INPUT_BASE.glob('**/metadata.txt'))
wavs_candidates = list(INPUT_BASE.glob('**/wavs'))

if not metadata_candidates:
    # List what's actually in /kaggle/input for debugging
    print('❌ metadata.txt not found! Contents of /kaggle/input:')
    !find /kaggle/input -maxdepth 3 -type f | head -20
    raise FileNotFoundError('metadata.txt not found in /kaggle/input')

metadata_src = metadata_candidates[0]
wavs_src = wavs_candidates[0] if wavs_candidates else metadata_src.parent / 'wavs'
print(f'Found metadata: {metadata_src}')
print(f'Found wavs dir: {wavs_src}')

# Create symlink to wavs directory
wavs_link = DATASET_DIR / 'wavs'
if wavs_link.exists() or wavs_link.is_symlink():
    wavs_link.unlink()
wavs_link.symlink_to(wavs_src)

# Copy metadata.txt as metadata.csv (same pipe-delimited format)
metadata_dst = DATASET_DIR / 'metadata.csv'

# Read and validate
lines = metadata_src.read_text().strip().split('\n')
print(f'\nTotal utterances: {len(lines)}')
print(f'First 3 entries:')
for line in lines[:3]:
    parts = line.split('|')
    print(f'  {parts[0]} → {parts[1][:60]}...' if len(parts[1]) > 60 else f'  {parts[0]} → {parts[1]}')

# Filter: only include lines where the WAV file actually exists
valid_lines = []
missing = 0
for line in lines:
    filename = line.split('|')[0]
    wav_path = wavs_src / filename
    if wav_path.exists() and wav_path.stat().st_size > 0:
        valid_lines.append(line)
    else:
        missing += 1

metadata_dst.write_text('\n'.join(valid_lines) + '\n')
print(f'\n✅ Valid utterances: {len(valid_lines)} (skipped {missing} missing files)')
print(f'Dataset directory: {DATASET_DIR}')

## 3. Preprocess (Phonemize + Normalize Audio)

Since espeak-ng does NOT support Iu Mienh, we use `--phoneme-type text` which treats each character/codepoint as a phoneme. This works well for languages with consistent orthography like Iu Mienh.

In [ ]:
PREPROCESSED_DIR = WORK_DIR / 'preprocessed'

!python3 -m piper_train.preprocess \
    --input-dir {DATASET_DIR} \
    --output-dir {PREPROCESSED_DIR} \
    --language ium \
    --sample-rate 22050 \
    --dataset-format ljspeech \
    --single-speaker \
    --phoneme-type text \
    --max-workers 4

# Verify output
config_path = PREPROCESSED_DIR / 'config.json'
dataset_path = PREPROCESSED_DIR / 'dataset.jsonl'

import json
config = json.loads(config_path.read_text())
num_lines = sum(1 for _ in open(dataset_path))

print(f'\n✅ Preprocessing complete!')
print(f'  Config: {config_path}')
print(f'  Dataset: {num_lines} utterances')
print(f'  Phoneme type: {config["phoneme_type"]}')
print(f'  Sample rate: {config["audio"]["sample_rate"]}')
print(f'  Num speakers: {config["num_speakers"]}')

## 4. Train the Model

Training a medium-quality VITS model. With ~5,900 utterances (~18 hours), expect:
- **P100 GPU**: ~2-3 min/epoch → 300 epochs ≈ 10-15 hours
- **T4 GPU**: ~3-4 min/epoch → 300 epochs ≈ 15-20 hours

Kaggle gives 30 hrs/week of GPU. You may need to:
1. Train for 100 epochs first, save checkpoint
2. Resume training later with `--resume_from_checkpoint`

**Tip:** Start with 100 epochs, listen to samples, then continue if quality isn't good enough.

In [ ]:
# === TRAINING CONFIGURATION ===
MAX_EPOCHS = 100        # Start with 100, increase later if needed
CHECKPOINT_EVERY = 25   # Save every 25 epochs
BATCH_SIZE = 16         # Reduce to 8 if you get OOM errors
QUALITY = 'medium'      # x-low (fast/small), medium (balanced), high (slow/best)

# Resume from checkpoint if available
resume_flag = ''
checkpoints = sorted(OUTPUT_DIR.glob('**/*.ckpt'))
if checkpoints:
    latest_ckpt = checkpoints[-1]
    resume_flag = f'--resume_from_checkpoint {latest_ckpt}'
    print(f'🔄 Resuming from: {latest_ckpt}')
else:
    print('🆕 Starting fresh training')

print(f'\nConfig: {MAX_EPOCHS} epochs, batch_size={BATCH_SIZE}, quality={QUALITY}')
print(f'Checkpoints saved every {CHECKPOINT_EVERY} epochs to {OUTPUT_DIR}')
print('\n--- Training starts below ---\n')

In [ ]:
!python3 -m piper_train \
    --dataset-dir {PREPROCESSED_DIR} \
    --accelerator gpu \
    --devices 1 \
    --batch-size {BATCH_SIZE} \
    --validation-split 0.05 \
    --max_epochs {MAX_EPOCHS} \
    --checkpoint-epochs {CHECKPOINT_EVERY} \
    --quality {QUALITY} \
    --default_root_dir {OUTPUT_DIR} \
    --precision 32 \
    {resume_flag}

## 5. Export to ONNX

Convert the trained checkpoint to a lightweight ONNX model (~50-80MB) that Piper can use for CPU inference.

In [ ]:
# Find the best/latest checkpoint
checkpoints = sorted(OUTPUT_DIR.glob('**/*.ckpt'))
if not checkpoints:
    print('❌ No checkpoints found! Training may not have completed.')
else:
    latest_ckpt = checkpoints[-1]
    onnx_path = WORK_DIR / 'iu-mienh.onnx'
    
    print(f'Exporting checkpoint: {latest_ckpt}')
    print(f'Output: {onnx_path}')
    
    !python3 -m piper_train.export_onnx \
        {latest_ckpt} \
        {onnx_path}
    
    # Copy config.json alongside the ONNX model (Piper needs both)
    config_out = WORK_DIR / 'iu-mienh.onnx.json'
    shutil.copy2(PREPROCESSED_DIR / 'config.json', config_out)
    
    if onnx_path.exists():
        size_mb = onnx_path.stat().st_size / (1024*1024)
        print(f'\n✅ Export successful!')
        print(f'  Model: {onnx_path} ({size_mb:.1f} MB)')
        print(f'  Config: {config_out}')
    else:
        print('\n❌ Export failed')

## 6. Test the Model

Generate a sample audio clip to verify the model works.

In [ ]:
import IPython.display as ipd
import numpy as np

onnx_path = WORK_DIR / 'iu-mienh.onnx'

if onnx_path.exists():
    # Test with a simple Iu Mienh sentence
    test_text = 'Tin-Hungh hnamv baamh mienh.'
    test_wav = WORK_DIR / 'test_output.wav'
    
    !echo "{test_text}" | python3 -m piper_train.infer_onnx \
        --model {onnx_path} \
        --config {WORK_DIR / 'iu-mienh.onnx.json'} \
        --output-file {test_wav}
    
    if test_wav.exists():
        print(f'✅ Generated: {test_text}')
        ipd.display(ipd.Audio(str(test_wav)))
    else:
        print('❌ Inference failed')
else:
    print('No ONNX model found. Run export first.')

## 7. Download the Model

Download these two files to your local machine:
- `iu-mienh.onnx` — The model (~50-80MB)
- `iu-mienh.onnx.json` — The config (phoneme map, etc.)

Place them in your translator project's `tts-model/` directory.

In [ ]:
# Create a zip for easy download
import zipfile

zip_path = Path('/kaggle/working/iu-mienh-tts-model.zip')
onnx_path = WORK_DIR / 'iu-mienh.onnx'
config_path = WORK_DIR / 'iu-mienh.onnx.json'

if onnx_path.exists():
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        zf.write(onnx_path, 'iu-mienh.onnx')
        zf.write(config_path, 'iu-mienh.onnx.json')
    
    size_mb = zip_path.stat().st_size / (1024*1024)
    print(f'✅ Download ready: {zip_path} ({size_mb:.1f} MB)')
    print(f'\nIn Kaggle: Click the output tab → download iu-mienh-tts-model.zip')
else:
    print('No model to zip. Complete training and export first.')

## Troubleshooting

### Out of Memory (OOM)
Reduce `BATCH_SIZE` to 8 or 4 in cell 4.

### Training too slow
- Use `QUALITY = 'x-low'` for faster training (smaller model)
- Reduce dataset size by filtering metadata.csv to fewer lines

### Resume after Kaggle timeout
Checkpoints are saved in `/kaggle/working/`. When you restart:
1. The resume logic in cell 4 will auto-detect the latest `.ckpt`
2. Increase `MAX_EPOCHS` to continue training further

### Model sounds robotic/bad
- Train longer (300+ epochs)
- Check that `metadata.txt` text matches audio accurately
- Filter out utterances where silence detection split poorly

### After downloading the model
On your local machine, test with:
```bash
pip install piper-tts
echo 'Tin-Hungh hnamv baamh mienh.' | piper --model iu-mienh.onnx --output_file test.wav
```